# SAUET Digital Wallet - Stress Test

End-to-end: cài đặt → clone repo GitHub → build SUT → setup MySQL → chạy test
(smoke/load/stress) → dump DB → reconcile → visualize.

**Yêu cầu**:
- Kaggle Settings: **Internet = On**, Accelerator = None.
- Repo GitHub chứa 2 thư mục: `digitalwallet-system/`,
  `digitalwallet-stress-test/`.

**Thứ tự cell**: chạy tuần tự Cell 2 → 13. Mỗi cell idempotent hoặc tự kiểm tra
điều kiện (đã cài, đã clone, đã chạy).

## Make sure (Install (JDK 21, MySQL, k6, git)

In [ ]:
%%bash
set -euo pipefail
LOG=/tmp/install.log; : > "$LOG"

# Chặn apt start service (container không systemd).
printf '#!/bin/sh\nexit 101\n' > /usr/sbin/policy-rc.d && chmod +x /usr/sbin/policy-rc.d

echo "[1/3] apt update + base packages"
apt-get update -qq >>"$LOG" 2>&1
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  -o Dpkg::Use-Pty=0 \
  openjdk-21-jdk-headless mysql-server git unzip curl jq >>"$LOG" 2>&1

echo "[2/3] k6 (pin v0.55.0)"
K6_VERSION=v0.55.0
if ! command -v k6 >/dev/null || [ "$(k6 version 2>/dev/null | awk '{print $2}')" != "v${K6_VERSION#v}" ]; then
  curl -sSL "https://github.com/grafana/k6/releases/download/${K6_VERSION}/k6-${K6_VERSION}-linux-amd64.tar.gz" \
    | tar -xz -C /tmp
  install -m 0755 "/tmp/k6-${K6_VERSION}-linux-amd64/k6" /usr/local/bin/k6
fi

echo "[3/3] Python deps"
pip install -q --disable-pip-version-check psutil pandas matplotlib

echo "--- Verify ---"
java -version 2>&1 | head -1
mysqld --version
k6 version
python -c "import psutil, pandas, matplotlib; print('py ok')"

## Clone 

sửa REPO_URL nếu cần

In [ ]:
%%bash
set -euo pipefail
REPO_URL="${REPO_URL:-https://github.com/vllddlvh/SAUETDigitalWallet.git}"
REPO_BRANCH="${REPO_BRANCH:-""}"          # "" = default branch của remote
REPO_DIR="${REPO_DIR:-/tmp/SAUETDigitalWallet}"
MARKER=/tmp/repo_state.txt

# State = "<url>\n<branch>" (branch rỗng nếu dùng default)
CUR_STATE=$(printf '%s\n%s' "$REPO_URL" "$REPO_BRANCH")
PREV_STATE=""
[ -f "$MARKER" ] && PREV_STATE=$(cat "$MARKER")

if [ -d "$REPO_DIR/.git" ] && [ "$CUR_STATE" = "$PREV_STATE" ]; then
  echo "Cùng URL+branch đã clone → fetch cập nhật"
  git -C "$REPO_DIR" fetch --depth 1 origin
  # Xác định ref cần reset:
  if [ -n "$REPO_BRANCH" ]; then
    REF="origin/$REPO_BRANCH"
  else
    REF=$(git -C "$REPO_DIR" rev-parse origin/HEAD 2>/dev/null \
          || git -C "$REPO_DIR" rev-parse origin/main 2>/dev/null \
          || git -C "$REPO_DIR" rev-parse origin/master)
  fi
  git -C "$REPO_DIR" reset --hard "$REF"
else
  echo "Clone mới (URL/branch thay đổi hoặc chưa có repo)"
  rm -rf "$REPO_DIR"
  if [ -n "$REPO_BRANCH" ]; then
    git clone --depth 1 --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_DIR"
  else
    git clone --depth 1 "$REPO_URL" "$REPO_DIR"
  fi
  printf '%s\n%s' "$REPO_URL" "$REPO_BRANCH" > "$MARKER"
fi

echo "--- Trạng thái repo ---"
echo "URL    : $REPO_URL"
echo "Branch : ${REPO_BRANCH:-<default>}"
git -C "$REPO_DIR" rev-parse --abbrev-ref HEAD
git -C "$REPO_DIR" log -1 --oneline

echo "--- Cấu trúc repo ---"
ls -la "$REPO_DIR"
for d in digitalwallet-docs digitalwallet-system digitalwallet-stress-test; do
  [ -d "$REPO_DIR/$d" ] || { echo "THIẾU: $d"; exit 1; }
done
echo "OK: đủ 3 thư mục."
echo "$REPO_DIR" > /tmp/repo_dir

## Setup

In [ ]:
%%bash
# MySQL

set -euo pipefail
REPO_DIR=$(cat /tmp/repo_dir)
export SUT_DIR="$REPO_DIR"
export SUT_DIR_FILE=/tmp/sut_dir

bash "$REPO_DIR/digitalwallet-stress-test/scripts/10_mysql.sh"

In [ ]:
%%bash
# SUT - DigitalWallet

set -euo pipefail
REPO_DIR=$(cat /tmp/repo_dir)
export SUT_DIR="$REPO_DIR"
export SUT_DIR_FILE=/tmp/sut_dir

bash "$REPO_DIR/digitalwallet-stress-test/scripts/20_build_sut.sh"

## Chạy test

Chọn `PROFILE` = `smoke` (3 min) / `load` (32 min) / `stress` (~30 min).

`orchestrator.run` thực hiện tuần tự:
1. `90_reset_db.sh` — kill app cũ + drop/tạo DB.
2. `--skip-build` → dùng jar đã build ở Cell 6.
3. `30_start_app.sh` — chạy jar, chờ HTTP ready.
4. `sampler.py` — CPU/RAM/JVM/MySQL → `sampler.csv`.
5. `k6 run` — sinh `k6_raw.csv` + `k6_summary.json`.
6. `80_dump_db.sh --mode=all` + `--mode=violated`.
7. `reconcile.py` — summary JSON/Markdown.

In [ ]:
import os, sys, subprocess
from datetime import datetime

# ==== Chọn profile ở đây ====
PROFILE = 'smoke'          # 'smoke' | 'load' | 'stress'
# ===========================

REPO_DIR   = open('/tmp/repo_dir').read().strip()
STRESS_DIR = f'{REPO_DIR}/digitalwallet-stress-test'
OUT_DIR    = f'/kaggle/working/results/{PROFILE}_{datetime.now().strftime("%Y%m%dT%H%M%S")}'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'PROFILE = {PROFILE}')
print(f'OUT_DIR = {OUT_DIR}')

env = os.environ.copy()
env.update({
    'SUT_DIR':       REPO_DIR,
    'SUT_DIR_FILE':  '/tmp/sut_dir',
    'RESULTS_DIR':   '/kaggle/working/results',
    'PROFILE':       PROFILE,
    'BASE_URL':      'http://localhost:8080',
})

os.chdir(STRESS_DIR)
cmd = [sys.executable, '-m', 'orchestrator.run',
       '--profile', PROFILE,
       '--skip-build',
       '--out', OUT_DIR]
print('$', ' '.join(cmd))
result = subprocess.run(cmd, env=env, check=False)
print(f'orchestrator exit = {result.returncode}')
if result.returncode == 2:
    print('LƯU Ý: k6 đã abort do stopCondition (threshold vượt ngưỡng) — đúng thiết kế stress.')

## Kết quả

Hiển thị summary Markdown + toàn bộ plot. Nếu có `invariant_violation`,
bảng lỗi ở cuối `reconcile_summary.md` liệt kê `wallet_id` để đối chiếu
với `violated_accounts.csv` / `violated_transactions.csv`.

In [ ]:
from IPython.display import display, Markdown, Image
import glob, os

# Summary
md_path = f'{OUT_DIR}/reconcile_summary.md'
if os.path.exists(md_path):
    with open(md_path) as f:
        display(Markdown(f.read()))
else:
    print('Không thấy reconcile_summary.md')

# Plots
pngs = sorted(glob.glob(f'{OUT_DIR}/plots/*.png'))
print(f'\n{len(pngs)} plots:')
for png in pngs:
    print('—', os.path.basename(png))
    display(Image(filename=png))

## Ending

- Copy kết quả về `/kaggle/working/results_final/` để commit.
- Kill app + mysqld (Kaggle commit mode dừng mọi background process khi cell cuối
  kết thúc, nhưng kill chủ động để tránh rò rỉ trong notebook session).

Chỉ chạy cell này khi đã xem xong kết quả. Nếu chạy tiếp một test khác, **không**
chạy Cell 13 — quay lại Cell 7 để đổi `PROFILE` và chạy Cell 8.

In [ ]:
import os, shutil, subprocess

dest = '/kaggle/working/results_final'
src = OUT_DIR
if os.path.abspath(src) != os.path.abspath(dest):
    shutil.copytree(src, dest, dirs_exist_ok=True)
    print(f'copied: {src} → {dest}')

print('\nFiles trong results_final:')
for f in sorted(os.listdir(dest)):
    p = os.path.join(dest, f)
    if os.path.isfile(p):
        print(f'  {f:40s} {os.path.getsize(p)/1024:10.1f} KB')
    else:
        print(f'  {f}/')

# Kill app + mysqld
subprocess.run(['bash', '-c', 'kill $(cat /tmp/app.pid) 2>/dev/null || true'])
subprocess.run(['bash', '-c', 'pkill -f "mysqld --user=root" 2>/dev/null || true'])
print('\ncleanup done.')